In [3]:
import torch

In [4]:
import torchvision

In [5]:
# install dependencies: 
!pip install pyyaml==5.1 ## human-readable data-serialization language
!gcc --version

     |████████████████████████████████| 276kB 8.2MB/s 
  Created wheel for pyyaml: filename=PyYAML-5.1-cp37-cp37m-linux_x86_64.whl size=44074 sha256=308d2148bbb55c454404528ac58b86271af0104e8d6f5201e1684066c9afcfa2
  Stored in directory: /root/.cache/pip/wheels/ad/56/bc/1522f864feb2a358ea6f1a92b4798d69ac783a28e80567a18b
Successfully built pyyaml
  Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13
gcc (Ubuntu 7.5.0-3ubuntu1~18.04) 7.5.0
Copyright (C) 2017 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [6]:
# install detectron2: (Colab has CUDA 10.1 + torch 1.8)
# See https://detectron2.readthedocs.io/tutorials/install.html for instructions
import torch
assert torch.__version__.startswith("1.8")   # need to manually install torch 1.8 if Colab changes its default version
!pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu101/torch1.8/index.html
exit(0)  # After installation, you need to "restart runtime" in Colab. This line can also restart runtime

Looking in links: https://dl.fbaipublicfiles.com/detectron2/wheels/cu101/torch1.8/index.html
     |████████████████████████████████| 6.2MB 1.7MB/s 
     |████████████████████████████████| 51kB 4.7MB/s 
  Created wheel for fvcore: filename=fvcore-0.1.3.post20210317-cp37-none-any.whl size=58543 sha256=001bfde4c732318ef75f94c1795e7e22ad286b137da68844984636fb428d5b2d
  Stored in directory: /root/.cache/pip/wheels/d2/ee/3a/5c531df777c03d8c67f22c65f97d6f75321087482d05a9b218
Successfully built fvcore


In [2]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

import matplotlib

# Model

In [3]:
cfg = get_cfg()
# add project-specific config (e.g., TensorMask) here if you're not running a model in detectron2's core library
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # set threshold for this model
# Find a model from detectron2's model zoo. You can use the https://dl.fbaipublicfiles... url as well
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml")
predictor = DefaultPredictor(cfg)

model_final_2d9806.pkl: 431MB [00:06, 67.9MB/s]                           


# Download Frames

In [ ]:
!wget https://github.com/gkioxari/aims2020_visualrecognition/releases/download/v1.0/videoclip.zip 
!unzip videoclip.zip

# Fetch & Sort Frames

In [5]:
import cv2
import numpy as np
import glob

images_array = []

# use glob to fetch the images
images = glob.glob('/content/clip/*.jpg')

# sort the images 
images.sort(key= lambda x : int(x.split('/')[-1].split('.')[0]))


# Video Tracker 

**take as input list of sorted frames (images) and output video**

In [10]:
def my_tracker(images):
  # start by the first frame
  image = cv2.imread(images[0])
  # predict the image
  outputs = predictor(image)
  # boxes in the first frame
  previous_boxes = outputs['instances'].pred_boxes
  # find the predicted classes (indexes)
  pred_classes = outputs['instances'].pred_classes
  # find the masks 
  pred_masks = outputs['instances'].pred_masks

  # labels
  thing_classes = MetadataCatalog.get(cfg.DATASETS.TRAIN[0]).thing_classes
  # labels for predicted classes
  predicted_classes_labels = list(map(lambda x: thing_classes[x], pred_classes.cpu().numpy()))
  
  # number of classes to use it for coloring the boxes
  num_of_classes = len(predicted_classes_labels)

  # colors from matplotlib 
  Colors = list(matplotlib.colors.cnames.keys())
  # set colors to the first frame
  previous_colors = Colors[-1:-1*(num_of_classes)*4:-4]

  # use visualizer to draw the boxes and give them colors and masks
  v = Visualizer(image, scale=1.2)
  out = v.overlay_instances(boxes=previous_boxes.tensor.cpu().numpy(), masks=pred_masks.cpu().numpy(), labels=predicted_classes_labels, assigned_colors=previous_colors)
  # show the predicted Frame
  cv2_imshow(out.get_image())

  # append the predicted frames including the boxes
  frames = []
  frames.append(out.get_image())



  



In [11]:
my_tracker(images)